In [1]:
import sys
sys.path.append("../src")
import copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from variables import ajouter_contexte
from evaluation import scores_classification, seuil_f1

h = pd.read_parquet("../data/processed/arcep_2022_2025.parquet")
web = h.query("protocole == 'WEB'").copy()
web["echec"] = (1 - web["succes"]).astype(int)
web = ajouter_contexte(web)

NUMERIQUES = ["rsrp", "heure", "echec_prec"]
BOOLEENNES = ["rsrp_absent", "prec_absent"]
CATEGORIES = ["operator", "zone", "situation"]
X = web[NUMERIQUES + BOOLEENNES + CATEGORIES]
y = web["echec"].to_numpy(dtype="float32")
annee = web["annee"]
entr, val, test = (annee <= 2023).to_numpy(), (annee == 2024).to_numpy(), (annee == 2025).to_numpy()

prep = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("echelle", StandardScaler())]), NUMERIQUES),
    ("bool", "passthrough", BOOLEENNES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIES),
])
prep.fit(X[entr])                                   # appris sur l'entraînement seulement

X_entr = torch.tensor(prep.transform(X[entr]).astype("float32"))
X_val  = torch.tensor(prep.transform(X[val]).astype("float32"))
X_test = torch.tensor(prep.transform(X[test]).astype("float32"))
y_entr = torch.tensor(y[entr]).unsqueeze(1)
y_val  = torch.tensor(y[val]).unsqueeze(1)

print(X_entr.shape, X_val.shape, X_test.shape)
print(prep.get_feature_names_out())

torch.Size([750585, 16]) torch.Size([248289, 16]) torch.Size([253260, 16])
['num__rsrp' 'num__heure' 'num__echec_prec' 'bool__rsrp_absent'
 'bool__prec_absent' 'cat__operator_Bouygues' 'cat__operator_Free'
 'cat__operator_Orange' 'cat__operator_SFR' 'cat__zone_Zones denses'
 'cat__zone_Zones intermediaires' 'cat__zone_Zones rurales'
 'cat__zone_Zones touristiques' 'cat__situation_Incar'
 'cat__situation_Indoor' 'cat__situation_Outdoor']


In [2]:
torch.manual_seed(0)
lots = DataLoader(TensorDataset(X_entr, y_entr), batch_size=512, shuffle=True)

modele = nn.Sequential(
    nn.Linear(16, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
print(sum(p.numel() for p in modele.parameters()), "paramètres")

taux_echec = y_entr.mean().item()
perte = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - taux_echec) / taux_echec]))
optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3, weight_decay=1e-5)

historique = []
meilleure, meilleurs_poids, patience, attente = float("inf"), None, 5, 0
for epoque in range(1, 31):
    modele.train()
    total = 0
    for xb, yb in lots:
        optimiseur.zero_grad()
        l = perte(modele(xb), yb)
        l.backward()
        optimiseur.step()
        total += l.item() * len(xb)

    modele.eval()
    with torch.no_grad():
        l_val = perte(modele(X_val), y_val).item()
    historique.append((total / len(X_entr), l_val))
    print(f"époque {epoque:2d}   perte entraînement {total / len(X_entr):.4f}   perte validation {l_val:.4f}")

    if l_val < meilleure:
        meilleure, meilleurs_poids, attente = l_val, copy.deepcopy(modele.state_dict()), 0
    else:
        attente += 1
        if attente >= patience:
            print("arrêt précoce")
            break

modele.load_state_dict(meilleurs_poids)

3201 paramètres
époque  1   perte entraînement 0.5979   perte validation 0.4814
époque  2   perte entraînement 0.5692   perte validation 0.4762
époque  3   perte entraînement 0.5673   perte validation 0.4648
époque  4   perte entraînement 0.5656   perte validation 0.4667
époque  5   perte entraînement 0.5647   perte validation 0.4658
époque  6   perte entraînement 0.5638   perte validation 0.4711
époque  7   perte entraînement 0.5633   perte validation 0.4659
époque  8   perte entraînement 0.5633   perte validation 0.4759
arrêt précoce


<All keys matched successfully>

In [3]:
with torch.no_grad():
    p_val = torch.sigmoid(modele(X_val)).squeeze(1).numpy()
    p_test = torch.sigmoid(modele(X_test)).squeeze(1).numpy()

seuil = seuil_f1(y[val], p_val)
resultats = pd.DataFrame({
    "MLP (validation 2024)": scores_classification(y[val], p_val, seuil),
    "MLP (test 2025)":       scores_classification(y[test], p_test, seuil),
}).T
print("seuil retenu :", round(float(seuil), 3))
resultats

seuil retenu : 0.887


,precision,rappel,f1,ap
MLP (validation 2024),0.785,0.717,0.749,0.801
MLP (test 2025),0.771,0.704,0.736,0.789


In [4]:
def entrainer_mlp(graine, patience=5, max_epoques=30):
    torch.manual_seed(graine)
    lots = DataLoader(TensorDataset(X_entr, y_entr), batch_size=512, shuffle=True)
    modele = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Dropout(0.2),
                           nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    taux = y_entr.mean().item()
    perte = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - taux) / taux]))
    optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3, weight_decay=1e-5)
    meilleure, meilleurs_poids, attente = float("inf"), None, 0
    for epoque in range(1, max_epoques + 1):
        modele.train()
        for xb, yb in lots:
            optimiseur.zero_grad()
            perte(modele(xb), yb).backward()
            optimiseur.step()
        modele.eval()
        with torch.no_grad():
            l_val = perte(modele(X_val), y_val).item()
        if l_val < meilleure:
            meilleure, meilleurs_poids, attente = l_val, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= patience:
                break
    modele.load_state_dict(meilleurs_poids)
    return modele

graines = {}
for g in [0, 1, 2]:
    m = entrainer_mlp(g)
    with torch.no_grad():
        pv = torch.sigmoid(m(X_val)).squeeze(1).numpy()
        pt = torch.sigmoid(m(X_test)).squeeze(1).numpy()
    s = seuil_f1(y[val], pv)
    graines[f"graine {g}"] = scores_classification(y[test], pt, s) | {"seuil": round(float(s), 3)}
    print(f"graine {g} terminée")

stabilite = pd.DataFrame(graines).T
stabilite.loc["moyenne"] = stabilite.mean().round(3)
stabilite.loc["écart-type"] = stabilite.iloc[:3].std().round(4)
stabilite.to_csv("../results/tables/mlp_web_test_2025.csv")
stabilite

graine 0 terminée
graine 1 terminée
graine 2 terminée


,precision,rappel,f1,ap,seuil
graine 0,0.7710,0.704,0.736,0.7890,0.8870
graine 1,0.7790,0.701,0.738,0.7900,0.8810
graine 2,0.7810,0.698,0.737,0.7890,0.8970
moyenne,0.7770,0.701,0.737,0.7890,0.8880
écart-type,0.0053,0.003,0.001,0.0006,0.0081
